# Setup · Build the multimodal, ACL-trimmed SharePoint index (REST)

This notebook creates the four Azure AI Search resources that power a **multimodal**,
**security-trimmed** index over a SharePoint document library, calling the Search
**REST API** directly so every payload is visible:

| # | Resource | REST call | What it does |
|---|----------|-----------|--------------|
| 1 | **Data source** | `PUT /datasources` | Connects to the SharePoint library and ingests Entra ACLs (`UserIds`/`GroupIds`). |
| 2 | **Index** | `PUT /indexes` | Fields + vector profiles + semantic config + `permissionFilterOption` (ACL trim). |
| 3 | **Skillset** | `PUT /skillsets` | Content Understanding (chunks + verbalized figures) + text & image vectors + projections. |
| 4 | **Indexer** | `PUT /indexers` | Cracks documents (`imageAction`), runs the skillset, writes text + image rows. |

Then we **run** the indexer, **poll** status, and run one **ACL-trimmed query**.

Everything is driven from `.env` — no hard-coded resource names or endpoints.

> **Prerequisites**: resources deployed (`infra/`) and app registration + RBAC done
> (`scripts/setup-app-registration.ps1`). You must be signed in with `az login` as an
> identity that has **Search Service Contributor** + **Search Index Data Contributor**
> on the search service. See the README **Permissions** section.

## 0 · Config + a tiny REST helper

We read every value from `.env`, acquire an Entra token for the Search data plane
(`https://search.azure.com`), and define a single `search_request()` helper. That's the
only abstraction in this notebook — every cell below just builds a JSON body and PUTs it.

In [1]:
import os, time, json, base64
from collections import Counter

import requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv()  # reads ../.env (or ./.env)

# --- Search service (data-plane REST) ---------------------------------------
SEARCH = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
API    = os.environ.get("AZURE_SEARCH_API_VERSION", "2026-05-01-preview")
PREFIX = os.environ.get("RESOURCE_PREFIX", "sharepoint")

# Resource names are derived from RESOURCE_PREFIX so you can stand up an isolated
# copy (e.g. RESOURCE_PREFIX=mytest) without touching an existing index.
DATASOURCE = f"{PREFIX}-nb7-ds"
INDEX      = f"{PREFIX}-nb7-index"
SKILLSET   = f"{PREFIX}-nb7-skillset"
INDEXER    = f"{PREFIX}-nb7-indexer"

# --- SharePoint data source --------------------------------------------------
SP_CONNECTION_STRING = os.environ["SHAREPOINT_CONNECTION_STRING"]
SP_CONTAINER         = os.environ.get("SHAREPOINT_CONTAINER_NAME", "defaultSiteLibrary")

# --- Azure OpenAI (text embeddings + Content Understanding verbalization) -----
AOAI_ENDPOINT    = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
EMBED_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
EMBED_MODEL      = os.environ["AZURE_OPENAI_EMBEDDING_MODEL"]
EMBED_DIMENSIONS = int(os.environ.get("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "3072"))
GPT_DEPLOYMENT   = os.environ["AZURE_OPENAI_GPT_DEPLOYMENT"]
GPT_MODEL        = os.environ["AZURE_OPENAI_GPT_MODEL"]

# --- Azure AI Vision (image embeddings) --------------------------------------
AI_SERVICES_ENDPOINT = os.environ["AZURE_AI_SERVICES_ENDPOINT"].rstrip("/")
VISION_MODEL_VERSION = os.environ.get("AZURE_AI_VISION_MODEL_VERSION", "2023-04-15")
VISION_DIMENSIONS    = 1024  # fixed by the Azure AI Vision multimodal embeddings API

# --- Keyless auth: Entra token for the Search data plane ---------------------
_cred = DefaultAzureCredential()
def token() -> str:
    return _cred.get_token("https://search.azure.com/.default").token

def search_request(method, path, body=None, extra_headers=None):
    """Call the Azure AI Search REST API. Raises with the server message on error."""
    url = f"{SEARCH}/{path}?api-version={API}"
    headers = {"Authorization": f"Bearer {token()}", "Content-Type": "application/json"}
    if extra_headers:
        headers.update(extra_headers)
    r = requests.request(method, url, headers=headers, json=body)
    if not r.ok:
        raise RuntimeError(f"{method} {path} -> {r.status_code}\n{r.text[:2000]}")
    if r.text and r.headers.get("content-type", "").startswith("application/json"):
        return r.json()
    return {}

print("search  :", SEARCH)
print("api ver :", API)
print("prefix  :", PREFIX)
print("objects :", DATASOURCE, "|", INDEX, "|", SKILLSET, "|", INDEXER)

search  : https://bcgbrain-search-wfhgpt.search.windows.net
api ver : 2026-05-01-preview
prefix  : sharepoint
objects : sharepoint-nb7-ds | sharepoint-nb7-index | sharepoint-nb7-skillset | sharepoint-nb7-indexer


## 1 · Data source — `PUT /datasources`

Connects the indexer to the SharePoint library. The key line for security trimming is
`indexerPermissionOptions: ["userIds", "groupIds"]`, which tells the indexer to pull each
item's Entra **user** and **group** ACLs so they can be indexed and later used to trim
query results per caller.

In [2]:
ds = {
    "name": DATASOURCE,
    "type": "sharepoint",
    "credentials": {"connectionString": SP_CONNECTION_STRING},
    "container": {"name": SP_CONTAINER, "query": None},
    # Ingest Entra ACLs alongside the content so the index can trim by identity.
    "indexerPermissionOptions": ["userIds", "groupIds"],
}
search_request("PUT", f"datasources/{DATASOURCE}", ds)
print("datasource ok:", DATASOURCE)

datasource ok: sharepoint-nb7-ds


## 2 · Index — `PUT /indexes`

Defines the searchable schema. Highlights:

- **`kind`** distinguishes `text` rows from `image` rows (both live in one index).
- **`content` / `contentVector`** — verbalized text + its 3072-dim Azure OpenAI embedding.
- **`imageData` / `imageVector`** — base64 image bytes (for rendering) + 1024-dim Azure AI
  Vision embedding (for text→image search).
- **`UserIds` / `GroupIds`** carry `permissionFilter` and, together with
  **`permissionFilterOption: "enabled"`**, enable **ACL trimming**: a query only returns
  rows the caller is allowed to see (see the query cell for the required header).

In [3]:
index = {
    "name": INDEX,
    "fields": [
        {"name": "id", "type": "Edm.String", "key": True, "analyzer": "keyword",
         "searchable": True, "filterable": True, "sortable": True, "retrievable": True},
        {"name": "parent_id", "type": "Edm.String", "filterable": True, "retrievable": True},
        {"name": "parent_id_img", "type": "Edm.String", "filterable": True, "retrievable": True},
        {"name": "kind", "type": "Edm.String", "filterable": True, "facetable": True, "retrievable": True},
        {"name": "content", "type": "Edm.String", "searchable": True, "analyzer": "standard.lucene",
         "retrievable": True},
        {"name": "contentVector", "type": "Collection(Edm.Single)", "searchable": True,
         "retrievable": False, "dimensions": EMBED_DIMENSIONS, "vectorSearchProfile": "text-profile"},
        {"name": "imageVector", "type": "Collection(Edm.Single)", "searchable": True,
         "retrievable": False, "dimensions": VISION_DIMENSIONS, "vectorSearchProfile": "image-profile"},
        {"name": "imageData", "type": "Edm.String", "retrievable": True,
         "searchable": False, "filterable": False, "sortable": False, "facetable": False},
        {"name": "page", "type": "Edm.Int32", "filterable": True, "sortable": True,
         "facetable": True, "retrievable": True},
        {"name": "pageTo", "type": "Edm.Int32", "filterable": True, "sortable": True,
         "facetable": True, "retrievable": True},
        {"name": "sourceFile", "type": "Edm.String", "searchable": True, "analyzer": "standard.lucene",
         "filterable": True, "facetable": True, "retrievable": True},
        {"name": "metadata_spo_item_id", "type": "Edm.String", "filterable": True, "retrievable": True},
        {"name": "webUrl", "type": "Edm.String", "retrievable": True},
        {"name": "metadata_spo_item_path", "type": "Edm.String", "retrievable": True},
        {"name": "lastModified", "type": "Edm.DateTimeOffset", "filterable": True, "sortable": True,
         "facetable": True, "retrievable": True},
        # permissionFilter fields — the basis for ACL trimming.
        {"name": "UserIds", "type": "Collection(Edm.String)", "permissionFilter": "userIds",
         "filterable": True, "retrievable": False},
        {"name": "GroupIds", "type": "Collection(Edm.String)", "permissionFilter": "groupIds",
         "filterable": True, "retrievable": False},
    ],
    "permissionFilterOption": "enabled",   # turn ACL trimming ON
    "vectorSearch": {
        "algorithms": [{"name": "hnsw", "kind": "hnsw",
                        "hnswParameters": {"m": 4, "efConstruction": 400, "metric": "cosine"}}],
        "vectorizers": [
            # Let the service embed text queries at search time (Azure OpenAI)...
            {"name": "aoai-vectorizer", "kind": "azureOpenAI",
             "azureOpenAIParameters": {"resourceUri": AOAI_ENDPOINT,
                                       "deploymentId": EMBED_DEPLOYMENT, "modelName": EMBED_MODEL}},
            # ...and embed text->image queries at search time (Azure AI Vision).
            {"name": "vision-vectorizer", "kind": "aiServicesVision",
             "aiServicesVisionParameters": {"resourceUri": AI_SERVICES_ENDPOINT,
                                            "authIdentity": None, "modelVersion": VISION_MODEL_VERSION}},
        ],
        "profiles": [
            {"name": "text-profile", "algorithm": "hnsw", "vectorizer": "aoai-vectorizer"},
            {"name": "image-profile", "algorithm": "hnsw", "vectorizer": "vision-vectorizer"},
        ],
    },
    "semantic": {
        "defaultConfiguration": "semantic-config",
        "configurations": [{"name": "semantic-config", "prioritizedFields": {
            "titleField": {"fieldName": "sourceFile"},
            "prioritizedContentFields": [{"fieldName": "content"}]}}],
    },
}
search_request("PUT", f"indexes/{INDEX}", index)
print("index ok:", INDEX)

index ok: sharepoint-nb7-index


## 3 · Skillset — `PUT /skillsets`

The enrichment pipeline. Four skills + two **index projections**:

1. **ContentUnderstandingSkill** — semantic chunking + page metadata + **inline figure
   verbalization** (each chunk's `content` embeds `![alt](figures/… "description")`, so
   diagrams are searchable as text). `modelName`/`modelDeployment` point at your GPT model.
2. **AzureOpenAIEmbeddingSkill** — embeds each chunk → `content_vector` (3072-dim).
3. **Vision VectorizeSkill** — embeds each **normalized image** → `image_vector` (1024-dim).
   It runs over `/document/normalized_images/*`, which is produced by the *indexer's*
   `imageAction` (next cell) — that array also carries the base64 image `data`.
4. **Two ConditionalSkills** — stamp `kind = "text"` / `kind = "image"`.

**Index projections** fan each source document out into many index rows:
- a **text** selector (one row per chunk), and
- an **image** selector (one row per normalized image), mapping `imageData ← …/data`,
  `imageVector ← …/image_vector`, and `page ← …/pageNumber`.

> **A note on `imagePath`.** Content Understanding can emit `imagePath` values (blob
> names like `…/normalized_images_13.jpg`), but this design has **no knowledge store**,
> so those paths point at nothing you can fetch. Images are carried as renderable bytes
> in `imageData` instead, so we omit `imagePath` from the schema and both projections
> entirely. (We also never map it under `normalized_images`, where no skill produces it —
> that would log *"Could not generate projection from input"*.)

In [4]:
# Shared metadata mapping (SharePoint fields + ACLs) applied to BOTH projections.
doc_meta = [
    {"name": "sourceFile", "source": "/document/metadata_spo_item_name"},
    {"name": "metadata_spo_item_id", "source": "/document/metadata_spo_item_id"},
    {"name": "webUrl", "source": "/document/metadata_spo_item_weburi"},
    {"name": "metadata_spo_item_path", "source": "/document/metadata_spo_item_path"},
    {"name": "lastModified", "source": "/document/metadata_spo_item_last_modified"},
    {"name": "UserIds", "source": "/document/metadata_user_ids"},
    {"name": "GroupIds", "source": "/document/metadata_group_ids"},
]

content_understanding = {
    "@odata.type": "#Microsoft.Skills.Util.ContentUnderstandingSkill",
    "name": "content-understanding",
    "description": "Semantic chunking + page metadata + image extraction + inline image verbalization.",
    "context": "/document",
    "extractionOptions": ["images", "locationMetadata"],
    "chunkingProperties": {"method": "semantic", "unit": "tokens",
                           "maximumLength": 800, "overlapLength": 0},
    "modelName": GPT_MODEL,
    "modelDeployment": GPT_DEPLOYMENT,
    "inputs": [{"name": "file_data", "source": "/document/file_data"}],
    "outputs": [{"name": "text_sections", "targetName": "text_sections"}],
}

text_embed = {
    "@odata.type": "#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill",
    "name": "text-embed", "context": "/document/text_sections/*",
    "resourceUri": AOAI_ENDPOINT, "deploymentId": EMBED_DEPLOYMENT,
    "modelName": EMBED_MODEL, "dimensions": EMBED_DIMENSIONS,
    "inputs": [{"name": "text", "source": "/document/text_sections/*/content"}],
    "outputs": [{"name": "embedding", "targetName": "content_vector"}],
}

image_embed = {
    "@odata.type": "#Microsoft.Skills.Vision.VectorizeSkill",
    "name": "image-embed", "context": "/document/normalized_images/*",
    "modelVersion": VISION_MODEL_VERSION,
    "inputs": [{"name": "image", "source": "/document/normalized_images/*"}],
    "outputs": [{"name": "vector", "targetName": "image_vector"}],
}

kind_text = {
    "@odata.type": "#Microsoft.Skills.Util.ConditionalSkill", "name": "kind-text",
    "context": "/document/text_sections/*",
    "inputs": [{"name": "condition", "source": "= true"},
               {"name": "whenTrue", "source": "= 'text'"},
               {"name": "whenFalse", "source": "= 'text'"}],
    "outputs": [{"name": "output", "targetName": "kind"}],
}
kind_image = {
    "@odata.type": "#Microsoft.Skills.Util.ConditionalSkill", "name": "kind-image",
    "context": "/document/normalized_images/*",
    "inputs": [{"name": "condition", "source": "= true"},
               {"name": "whenTrue", "source": "= 'image'"},
               {"name": "whenFalse", "source": "= 'image'"}],
    "outputs": [{"name": "output", "targetName": "kind"}],
}

skillset = {
    "name": SKILLSET,
    "description": "CU (semantic chunks + image extraction) -> text + image vectors; project ACLs.",
    "skills": [content_understanding, text_embed, image_embed, kind_text, kind_image],
    "cognitiveServices": {"@odata.type": "#Microsoft.Azure.Search.AIServicesByIdentity",
                          "subdomainUrl": AI_SERVICES_ENDPOINT},
    "indexProjections": {
        "selectors": [
            {   # one row per text chunk
                "targetIndexName": INDEX, "parentKeyFieldName": "parent_id",
                "sourceContext": "/document/text_sections/*",
                "mappings": [
                    {"name": "content", "source": "/document/text_sections/*/content"},
                    {"name": "contentVector", "source": "/document/text_sections/*/content_vector"},
                    {"name": "kind", "source": "/document/text_sections/*/kind"},
                    {"name": "page", "source": "/document/text_sections/*/locationMetadata/pageNumberFrom"},
                    {"name": "pageTo", "source": "/document/text_sections/*/locationMetadata/pageNumberTo"},
                ] + doc_meta,
            },
            {   # one row per normalized image (bytes come from the indexer's imageAction)
                "targetIndexName": INDEX, "parentKeyFieldName": "parent_id_img",
                "sourceContext": "/document/normalized_images/*",
                "mappings": [
                    {"name": "imageData", "source": "/document/normalized_images/*/data"},
                    {"name": "imageVector", "source": "/document/normalized_images/*/image_vector"},
                    {"name": "kind", "source": "/document/normalized_images/*/kind"},
                    {"name": "page", "source": "/document/normalized_images/*/pageNumber"},
                    {"name": "pageTo", "source": "/document/normalized_images/*/pageNumber"},
                ] + doc_meta,
            },
        ],
        "parameters": {"projectionMode": "skipIndexingParentDocuments"},
    },
}
search_request("PUT", f"skillsets/{SKILLSET}", skillset)
print("skillset ok:", SKILLSET)

skillset ok: sharepoint-nb7-skillset


## 4 · Indexer — `PUT /indexers`

Ties data source + skillset + index together. The critical setting is
**`imageAction: "generateNormalizedImages"`**: it cracks each document and emits
`/document/normalized_images/*` with base64 image **`data`** — the reliable source for
`imageData` (Content Understanding's own image output does *not* populate `data`).

Creating the indexer **auto-starts a run**.

In [5]:
indexer = {
    "name": INDEXER,
    "dataSourceName": DATASOURCE,
    "targetIndexName": INDEX,
    "skillsetName": SKILLSET,
    "parameters": {
        "batchSize": 1,
        "maxFailedItems": 5,
        "maxFailedItemsPerBatch": 5,
        "configuration": {
            "dataToExtract": "contentAndMetadata",
            "allowSkillsetToReadFileData": True,
            # Emit base64 image bytes so imageData is renderable (and image_vector has input).
            "imageAction": "generateNormalizedImages",
        },
    },
    "fieldMappings": [],
    "outputFieldMappings": [],
}
search_request("PUT", f"indexers/{INDEXER}", indexer)
print("indexer ok (auto-run started):", INDEXER)

indexer ok (auto-run started): sharepoint-nb7-indexer


## 5 · Poll the indexer — `GET /indexers/{name}/status`

Wait for the run to finish and print any errors/warnings. Two informational warnings are
expected and harmless: the **ACL folder-propagation** notice and the **`imageAction`
billing** notice. There should be **no** `errors` and **no** *"Could not generate
projection"* warnings.

In [6]:
for _ in range(90):
    st = search_request("GET", f"indexers/{INDEXER}/status")
    last = st.get("lastResult") or {}
    print(f"status={st.get('status'):<10} last={last.get('status')} "
          f"processed={last.get('itemsProcessed')} failed={last.get('itemsFailed')}")
    if (last.get("status") or "").lower() in ("success", "transientFailure".lower()) and last.get("endTime"):
        break
    time.sleep(20)

errors   = last.get("errors") or []
warnings = last.get("warnings") or []
print(f"\nerrors: {len(errors)}  |  warnings: {len(warnings)}")
for e in errors[:5]:
    print("  ERR :", e.get("key"), (e.get("errorMessage") or "")[:200])
for w in warnings[:5]:
    print("  WARN:", (w.get("message") or "")[:160])
proj = [w for w in warnings if "projection" in (w.get("message", "").lower())]
print("\nprojection warnings (should be 0):", len(proj))

status=running    last=None processed=None failed=None


status=running    last=None processed=None failed=None


status=running    last=inProgress processed=0 failed=0


status=running    last=inProgress processed=0 failed=0


status=running    last=inProgress processed=0 failed=0


status=running    last=success processed=3 failed=0

errors: 0  |  warnings: 2
  WARN: Folder propagation was automatically enabled because ACL ingestion is configured. Child items will be re-indexed when folder permissions change. To explicitly o
  WARN: ImageAction is set on the indexer and will incur an additional charge for image extraction.

projection warnings (should be 0): 0


## 6 · Verify — an **ACL-trimmed** query

The index has `permissionFilterOption: "enabled"`, so results are trimmed to what the
**caller** may see. To exercise that trim you must forward your identity in the
**`x-ms-query-source-authorization`** header (the same `search.azure.com` token works for
both `Authorization` and this header). Without it you'd only get documents shared with
*everyone*.

We confirm: total rows, the text/image split, that image rows carry non-empty
`imageData`, and that at least one chunk contains a verbalized figure marker.

In [7]:
hdr = {"x-ms-query-source-authorization": token()}   # forward caller identity => ACL trim
r = search_request("POST", f"indexes/{INDEX}/docs/search", {
    "search": "*", "count": True, "top": 100,
    "select": "kind,sourceFile,page,content,imageData",
}, extra_headers=hdr)

docs = r.get("value", [])
print("total rows (visible to you):", r.get("@odata.count"))
print("by kind:", dict(Counter(d["kind"] for d in docs)))

imgs = [d for d in docs if d["kind"] == "image"]
print("image rows with non-empty imageData:",
      sum(1 for d in imgs if (d.get("imageData") or "")), "/", len(imgs))

import re
verb = [d for d in docs if d["kind"] == "text"
        and re.search(r'!\[[^\]]*\]\([^)]*"[^)]*"\)', d.get("content") or "")]
print("text chunks with verbalized figures:", len(verb))

if imgs and imgs[0].get("imageData"):
    b = imgs[0]["imageData"]
    b = b.split(",", 1)[1] if b.startswith("data:") else b
    raw = base64.b64decode(b)
    print("sample image:", len(raw), "bytes | valid JPEG:", raw[:3] == b"\xff\xd8\xff")

total rows (visible to you): 89
by kind: {'text': 57, 'image': 32}
image rows with non-empty imageData: 32 / 32
text chunks with verbalized figures: 11
sample image: 950448 bytes | valid JPEG: True


---

### Done ✅

The datasource, index, skillset and indexer are live and the index is populated with
ACL-trimmed **text** and **image** rows.

**Next:** open **`notebooks/demo_retrieval_and_images.ipynb`** to run full-text, semantic,
and text→image vector searches and render the images inline.

**Re-run / iterate:** `POST /indexers/{INDEXER}/run` re-runs the pipeline. To build an
isolated copy, set a different `RESOURCE_PREFIX` in `.env` and re-run this notebook; delete
a copy with `DELETE /indexers|skillsets|indexes|datasources/{name}`.